In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [ ]:
import torch

# Define device globally
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [ ]:
import os
print(os.listdir(path))

['IMDB Dataset.csv']


In [ ]:
df = pd.read_csv(os.path.join(path, 'IMDB Dataset.csv'))
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.info()
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove special characters
    text = text.lower() # Convert to lowercase
    return text

df['cleaned_review'] = df['review'].apply(clean_text)
df.head()

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,1,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,1,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,1,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,0,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,petter matteis love in the time of money is a ...


In [ ]:
X = df['cleaned_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

Training set size: 40000 samples
Testing set size: 10000 samples


In [ ]:
!pip install transformers peft accelerate bitsandbytes -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.0 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade torchao -qqq

In [ ]:
!pip install --upgrade torchao -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.2 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade torchao>=0.1.7 -qqq

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

max_length = 128


X_train_tokenized = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=max_length, return_tensors='pt')
X_test_tokenized = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=max_length, return_tensors='pt')

print("Tokenization complete for training data. Example input_ids shape:", X_train_tokenized['input_ids'].shape)
print("Tokenization complete for testing data. Example input_ids shape:", X_test_tokenized['input_ids'].shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenization complete for training data. Example input_ids shape: torch.Size([40000, 128])
Tokenization complete for testing data. Example input_ids shape: torch.Size([10000, 128])


In [ ]:
# Train the LoRA model
lora_trainer.train()

# Evaluate the LoRA model
lora_results = lora_trainer.evaluate()
print(f"LoRA Model Accuracy: {lora_results['eval_accuracy']:.4f}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.382091,0.358894,0.837500
2,0.345859,0.345015,0.847600
3,0.323559,0.337691,0.848600


Training Loss,Validation Loss,Epoch,Accuracy
0.323559,0.337691,3,0.848600


LoRA Model Accuracy: 0.8486


### LoRA Model Predictions
Let's see how the LoRA fine-tuned model performs on some sample test data.

In [ ]:
import numpy as np


predictions = lora_trainer.predict(test_dataset)


predicted_labels = np.argmax(predictions.predictions, axis=1)

sentiment_map = {1: 'positive', 0: 'negative'}
true_sentiments = [sentiment_map[label] for label in y_test_np]
predicted_sentiments = [sentiment_map[label] for label in predicted_labels]

print("Sample Predictions:")
for i in range(5):
    print(f"\nReview: {X_test.iloc[i][:150]}...")
    print(f"Actual Sentiment: {true_sentiments[i]}")
    print(f"Predicted Sentiment: {predicted_sentiments[i]}")

Sample Predictions:

Review: i really liked this summerslam due to the look of the arena the curtains and just the look overall was interesting to me for some reason anyways this ...
Actual Sentiment: positive
Predicted Sentiment: negative

Review: not many television shows appeal to quite as many different kinds of fans like farscape doesi know youngsters and  years oldfans both male and female ...
Actual Sentiment: positive
Predicted Sentiment: positive

Review: the film quickly gets to a major chase scene with ever increasing destruction the first really bad thing is the guy hijacking steven seagal would have...
Actual Sentiment: negative
Predicted Sentiment: negative

Review: jane austen would definitely approve of this onegwyneth paltrow does an awesome job capturing the attitude of emma she is funny without being excessiv...
Actual Sentiment: positive
Predicted Sentiment: positive

Review: expectations were somewhat high for me when i went to see this movie after all i thought st

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.to(device)

print(f"Model loaded and moved to: {device}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to: cuda


In [ ]:
from torch.utils.data import Dataset

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# Convert y_train and y_test to numpy arrays for easier indexing
y_train_np = y_train.values
y_test_np = y_test.values

train_dataset = SentimentDataset(X_train_tokenized, y_train_np)
test_dataset = SentimentDataset(X_test_tokenized, y_test_np)

print("Custom datasets created.")

Custom datasets created.


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()


training_args = TrainingArguments(
    output_dir="./lora_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./lora_logs',
    logging_steps=100,
    save_steps=500,
    save_total_limit=2
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    return {"accuracy": accuracy}

lora_trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("LoRA model configured and Trainer instantiated.")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925
LoRA model configured and Trainer instantiated.
